In [1]:
from huggingface_hub import snapshot_download

snapshot_download("codelion/fineweb-edu-1B", repo_type="dataset", local_dir="./fineweb_data")
print("Downloaded!")

/home/awais/miniconda3/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 12 files: 100%|██████████| 12/12 [13:52<00:00, 69.39s/it]

Downloaded!


In [ ]:
#tokenized codelion/fineweb-edu-1B 
import tiktoken
import numpy as np
import os
import pandas as pd

enc=tiktoken.get_encoding("gpt2")
eot=enc._special_tokens["<|endoftext|>"]
data_dir = "/mnt/d/AI/GPT-2/fineweb_data/data"
token_count=0
train_count=0

train_file=open("/mnt/d/AI/GPT-2/fineweb_data/tokenized/train_tokens.bin","wb")
val_file=open("/mnt/d/AI/GPT-2/fineweb_data/tokenized/val_tokens.bin","wb")
for filename in sorted(os.listdir(data_dir)):
    if filename.endswith('.parquet'):
        filepath = os.path.join(data_dir, filename)
        print(f"Processing {filename}...")
        
        df = pd.read_parquet(filepath)
        
        for text in df['text']:
            tokens = np.array([eot]+enc.encode_ordinary(text), dtype=np.uint16)
            if token_count < 900_000_000:
                train_file.write(tokens.tobytes())
            else:
                val_file.write(tokens.tobytes())
            token_count += len(tokens)

train_file.close()
val_file.close()
print(f"{token_count} tokens saved!")

Processing train-00000-of-00010.parquet...
Processing train-00001-of-00010.parquet...
Processing train-00002-of-00010.parquet...
Processing train-00003-of-00010.parquet...
Processing train-00004-of-00010.parquet...
Processing train-00005-of-00010.parquet...
Processing train-00006-of-00010.parquet...
Processing train-00007-of-00010.parquet...
Processing train-00008-of-00010.parquet...
Processing train-00009-of-00010.parquet...
1000970542 tokens saved!


In [1]:
import tiktoken
import numpy as np
import os
import pandas as pd

enc=tiktoken.get_encoding("gpt2")
eot=enc._special_tokens["<|endoftext|>"]
data_dir = "/mnt/d/AI/GPT-2/fineweb_data/data 2"

all_file=open("/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/all_tokens.bin","ab")

for filename in sorted(os.listdir(data_dir)):
    if filename=='000_00002.parquet':
        filepath = os.path.join(data_dir, filename)
        print(f"Processing {filename}...")
        
        df = pd.read_parquet(filepath)
        print(f"  Tokenizing...")
        
        for text in df['text']:
            tokens = np.array([eot]+enc.encode_ordinary(text), dtype=np.uint16)
            all_file.write(tokens.tobytes())
        
        print(f"  {filename} done!")

Processing 000_00002.parquet...
  Tokenizing...
  000_00002.parquet done!


In [1]:
import numpy as np

filepath = "/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/all_tokens.bin"
tokens = np.fromfile(filepath, dtype=np.uint16)
print(f"Total tokens: {len(tokens):,}")

Total tokens: 2,257,517,672


In [2]:
import numpy as np

filepath = "/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/all_tokens.bin"
tokens = np.fromfile(filepath, dtype=np.uint16)

split_idx = int(len(tokens) * 0.9)  # 90/10 split
train_tokens = tokens[:split_idx]
val_tokens = tokens[split_idx:]

train_tokens.tofile("/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/train_tokens.bin")
val_tokens.tofile("/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/val_tokens.bin")

print(f"Train: {len(train_tokens):,}, Val: {len(val_tokens):,}")

Train: 2,031,765,904, Val: 225,751,768


In [1]:
import numpy as np

train_tokens = np.fromfile("/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/train_tokens.bin", dtype=np.uint16)
train_half = int(len(train_tokens) / 2)
train_part1 = train_tokens[:train_half]
train_part2 = train_tokens[train_half:]

val_tokens = np.fromfile("/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/val_tokens.bin", dtype=np.uint16)
val_half = int(len(val_tokens) / 2)
val_part1 = val_tokens[:val_half]
val_part2 = val_tokens[val_half:]

train_part1.tofile("/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/train_tokens_1.bin")
train_part2.tofile("/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/train_tokens_2.bin")
val_part1.tofile("/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/val_tokens_1.bin")
val_part2.tofile("/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/val_tokens_2.bin")

print(f"Train: {len(train_part1)}, {len(train_part2)}")
print(f"Val: {len(val_part1)}, {len(val_part2)}")

Train: 1015882952, 1015882952
Val: 112875884, 112875884


In [ ]:
#Trained on codelion/fineweb-edu-1B 
import torch
from torch.nn import functional as F
from torch.nn import ModuleDict
from dataclasses import dataclass
from transformers import GPT2LMHeadModel
import torch.nn as nn
import inspect
import math
import time
import numpy as np
import os

class CausalAttention(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.c_attn=nn.Linear(config.n_embd,3*config.n_embd)
                self.c_proj=nn.Linear(config.n_embd,config.n_embd)
                self.n_embd=config.n_embd
                self.n_head=config.n_head
                self.c_proj.NANOGPT_SCALE_INIT=1
                self.register_buffer("bias",torch.tril(torch.ones(config.block_size,config.block_size)).view(1,1,config.block_size,config.block_size))
        def forward(self,x):
                B,T,C=x.shape
                qkv=self.c_attn(x)
                q,k,v=qkv.split(self.n_embd,dim=2)
                q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
                k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
                v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
                
                # att=(q@k.transpose(-2,-1))*(1/math.sqrt(k.size(-1)))
                # att=att.masked_fill(self.bias[:,:,:T,:T]==0,float("-inf"))
                # att=F.softmax(att,dim=-1)
                # y=att@v
                y=F.scaled_dot_product_attention(q,k,v,is_causal=True)
                
                y=y.transpose(1,2).contiguous().view(B,T,C)
                y=self.c_proj(y)
                return y

class MLP(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.c_fc=nn.Linear(config.n_embd,4*config.n_embd)
                self.gelu=nn.GELU(approximate="tanh")
                self.c_proj=nn.Linear(4*config.n_embd,config.n_embd)
                self.c_proj.NANOGPT_SCALE_INIT=1

        def forward(self,x):
                x=self.c_fc(x)
                x=self.gelu(x)
                x=self.c_proj(x)
                return x

class Block(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.config=config
                self.ln_1=nn.LayerNorm(config.n_embd)
                self.attn=CausalAttention(config)
                self.ln_2=nn.LayerNorm(config.n_embd)
                self.mlp=MLP(config)
        def forward(self,x):
                x=x+self.attn(self.ln_1(x))
                x=x+self.mlp(self.ln_2(x))
                return x

@dataclass
class GPTConfig:
        vocab_size:int=50257
        block_size:int=1024
        n_head:int=12
        n_layer:int=12
        n_embd:int=768
class GPT(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.config=config
                self.transformer=ModuleDict(dict(
                        wte=nn.Embedding(config.vocab_size,config.n_embd),
                        wpe=nn.Embedding(config.block_size,config.n_embd),
                        h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
                        ln_f=nn.LayerNorm(config.n_embd),
                ))
                self.lm_head=nn.Linear(config.n_embd,config.vocab_size,bias=False)
                self.transformer.wte.weight=self.lm_head.weight
                self.apply(self.init_weights)
        def init_weights(self,module):
               std=0.02
               if isinstance(module,nn.Linear):
                      if hasattr(module, 'NANOGPT_SCALE_INIT'):
                                std*=(2*self.config.n_layer)**-0.5
                      torch.nn.init.normal_(module.weight,mean=0.0,std=std)
                      if module.bias is not None:
                          torch.nn.init.zeros_(module.bias)
               if isinstance(module,nn.Embedding):
                        torch.nn.init.normal_(module.weight,mean=0.0,std=std)
        def forward(self, idx, targets=None):
                B, T = idx.size()
                assert T <= self.config.block_size, f"Cannot forward sequence of length {T},block size is only {self.config.block_size}"
                pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
                pos_emb = self.transformer.wpe(pos)
                tok_emb = self.transformer.wte(idx)
                x = tok_emb + pos_emb
                for block in self.transformer.h:
                        x = block(x)
                x = self.transformer.ln_f(x)
                logits = self.lm_head(x)
                loss = None
                if targets is not None:
                        loss = F.cross_entropy(logits.view(-1, logits.size(-1)),targets.view(-1))
                return logits, loss
        def configure_optimizers(self,weight_decay,learning_rate):
                param_dict={pn:p for pn,p in self.named_parameters()}
                param_dict={pn:p for pn,p in param_dict.items() if p.requires_grad}
                decay_params = [p for n,p in param_dict.items() if p.dim() >= 2]
                no_decay_params = [p for n,p in param_dict.items() if p.dim() < 2]
                optim_groups = [
                        {"params": decay_params, "weight_decay": weight_decay},
                        {"params": no_decay_params, "weight_decay": 0.0},
                ]
                num_decay_params = sum(p.numel() for p in decay_params)
                num_no_decay_params = sum(p.numel() for p in no_decay_params)
                print(f"num decay params: {num_decay_params}, num no decay params: {num_no_decay_params}")
                fused_available='fused' in inspect.signature(torch.optim.AdamW).parameters
                use_fused = fused_available and torch.cuda.is_available()
                if use_fused:
                        print(f"using fused AdamW: {use_fused}")
                optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8, fused=use_fused)
                return optimizer
        #IF YOU WANT TO LOAD PRETRAINED GPT-2 WEIGHTS
        # @classmethod
        # def from_pretrained(cls, model_type):
        #         """Loads pretrained GPT-2 model weights from huggingface"""
        #         assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        #         print("loading weights from pretrained gpt: %s" % model_type)
        #         config_args = {
        #                 'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
        #                 'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
        #                 'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
        #                 'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        #         }[model_type]
        #         config_args['vocab_size']=50257
        #         config_args['block_size']=1024
        #         config = GPTConfig(**config_args)
        #         model = GPT(config)
        #         sd = model.state_dict()
        #         sd_keys = sd.keys()
        #         sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')]
        #         model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        #         sd_hf = model_hf.state_dict()
        #         sd_keys_hf = sd_hf.keys()
        #         sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')]
        #         sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')]
        #         transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        #         assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        #         for k in sd_keys_hf:
        #                 if any(k.endswith(w) for w in transposed):
        #                         assert sd_hf[k].shape[::-1]==sd[k].shape
        #                         with torch.no_grad():
        #                                 sd[k].copy_(sd_hf[k].t())
        #                 else:
        #                         assert sd_hf[k].shape == sd[k].shape
        #                         with torch.no_grad():
        #                                 sd[k].copy_(sd_hf[k])
        #         return model

# def load_tokens(filename):
#     npt = np.load(filename)
#     npt = npt.astype(np.int32)
#     ptt = torch.tensor(npt, dtype=torch.long)
#     return ptt

class DataLoaderLite:
    def __init__(self, B, T, process_rank=0, num_processes=1, split="train"):
        self.B = B
        self.T = T
        self.current_position = 0
        if split=="train":
                filepath="/mnt/d/AI/GPT-2/fineweb_data/tokenized/train_tokens.bin"
        else:
                filepath="/mnt/d/AI/GPT-2/fineweb_data/tokenized/val_tokens.bin"
        with open(filepath, "rb") as f:
                tokens_np= np.frombuffer(f.read(),dtype=np.uint16)
        self.tokens = torch.tensor(tokens_np,dtype=torch.long)

        #IF WE WANT TO USE MULTIPLE PROCESSES, WE CAN USE THIS CODE
        # self.process_rank = process_rank
        # self.num_processes = num_processes
        # assert split in {'train', 'val'}
        # data_root = "/mnt/d/AI/GPT-2/fineweb_data/tokenized"
        # shards = os.listdir(data_root)
        # shards = [s for s in shards if split in s]
        # shards = sorted(shards)
        # shards = [os.path.join(data_root, s) for s in shards]
        # self.shards = shards
        # assert len(shards) > 0, f"no shards found for split {split}"
        # if self.process_rank == 0:
        #     print(f"found {len(shards)} shards for split {split}")
        # self.reset()

#     def reset(self):
#         self.current_shard = 0
#         self.tokens = load_tokens(self.shards[self.current_shard])
#         self.current_position = self.B * self.T * self.process_rank

    def next_batch(self):
        B, T = self.B, self.T
        if self.current_position + (B * T + 1) > len(self.tokens):
                self.current_position = 0
        buf = self.tokens[self.current_position : self.current_position+B*T+1]
        x = (buf[:-1]).view(B, T) # inputs
        y = (buf[1:]).view(B, T) # targets
        # advance the position in the tensor
        self.current_position += B * T # * self.num_processes
        # if loading the next batch would be out of bounds, advance to next shard
        # if self.current_position + (B * T * self.num_processes + 1) > len(self.tokens):
        #     self.current_shard = (self.current_shard + 1) % len(self.shards)
        #     self.tokens = load_tokens(self.shards[self.current_shard])
        #     self.current_position = B * T * self.process_rank
        return x, y

B=16
T=1024
batch_size=65536
assert batch_size%(B*T)==0, "batch size must be divisible by B*T"
grad_accum_steps=batch_size//(B*T)
print(f"batch_size={batch_size}")
print(f"using grad_accum_steps={grad_accum_steps}")

max_lr=6e-4
min_lr=max_lr*0.1
warmup_steps=70
max_steps=1907
def lr_schedule(it):
        if it<warmup_steps:
                return max_lr*(it+1)/warmup_steps
        if it>max_steps:
                return min_lr
        decay_ratio=(it-warmup_steps)/(max_steps-warmup_steps)
        coeff=0.5*(1+math.cos(math.pi*decay_ratio))
        return min_lr+(max_lr-min_lr)*coeff

torch.set_float32_matmul_precision('high')
train_loader=DataLoaderLite(B=16, T=1024)
val_loader=DataLoaderLite(B=16, T=1024, split="val")
model=GPT(GPTConfig(vocab_size=50304))
model.to("cuda")
model=torch.compile(model)
optimizer=model.configure_optimizers(weight_decay=0.1,learning_rate=6e-4)
for step in range(max_steps):
        time1=time.time()
        optimizer.zero_grad()
        loss_accum=0.0
        for micro_step in range(grad_accum_steps):
                x,y=train_loader.next_batch()
                x,y=x.to("cuda"),y.to("cuda")
                with torch.autocast(device_type="cuda",dtype=torch.bfloat16):
                        logits,loss=model(x,y)
                loss=loss/grad_accum_steps
                loss_accum += loss.detach()
                loss.backward()
        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)               
        lr=lr_schedule(step)
        for param_group in optimizer.param_groups:
               param_group["lr"]=lr
        optimizer.step()
        if step % 5 == 0:
                torch.save(model.state_dict(), f"model_step_{step}.pt")
                val_loss = 0.0
                for _ in range(10):
                        x_val, y_val = val_loader.next_batch()
                        x_val, y_val = x_val.to("cuda"), y_val.to("cuda")
                        with torch.no_grad():
                                _, loss_val = model(x_val, y_val)
                        val_loss += loss_val.item()
                print(f"val_loss: {val_loss/10:.4f}")
        time2=time.time()
        torch.cuda.synchronize()
        tokens_per_sec = (train_loader.B * train_loader.T) / (time2 - time1)
        t=(time2-time1)*1000
        print(f"step {step} | loss {loss_accum.item():.4f} | norm {norm.item():.4f} | lr:{lr} | time: {t:.4f} ms")

/home/awais/miniconda3/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


batch_size=65536
using grad_accum_steps=4
num decay params: 124354560, num no decay params: 121344
using fused AdamW: True


W0425 23:34:37.569000 2968 site-packages/torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


val_loss: 10.5660
step 0 | loss 10.9732 | norm 14.7213 | lr:8.571428571428571e-06 | time: 36473.0077 ms
step 1 | loss 10.5588 | norm 10.4703 | lr:1.7142857142857142e-05 | time: 3488.4973 ms
step 2 | loss 10.2158 | norm 6.7151 | lr:2.5714285714285714e-05 | time: 3485.5006 ms
step 3 | loss 9.9341 | norm 4.5840 | lr:3.4285714285714284e-05 | time: 3487.5500 ms
step 4 | loss 9.7500 | norm 3.9263 | lr:4.285714285714285e-05 | time: 3489.3315 ms
val_loss: 9.6114
step 5 | loss 9.6943 | norm 2.9367 | lr:5.142857142857143e-05 | time: 15084.8086 ms
step 6 | loss 9.6125 | norm 2.5216 | lr:5.9999999999999995e-05 | time: 3502.2807 ms
step 7 | loss 9.6014 | norm 2.3750 | lr:6.857142857142857e-05 | time: 3516.0728 ms
step 8 | loss 9.4623 | norm 2.4826 | lr:7.714285714285714e-05 | time: 3518.2366 ms
step 9 | loss 9.4579 | norm 2.2866 | lr:8.57142857142857e-05 | time: 3514.6875 ms
val_loss: 9.2394
step 10 | loss 9.3379 | norm 2.1487 | lr:9.428571428571427e-05 | time: 14844.6033 ms
step 11 | loss 9.2529 |

In [1]:
#Trained on fineweb-edu-350B-0,1,2 .Parquets (1st half)
import torch
from torch.nn import functional as F
from torch.nn import ModuleDict
from dataclasses import dataclass
from transformers import GPT2LMHeadModel
import torch.nn as nn
import inspect
import math
import time
import numpy as np
import os

class CausalAttention(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.c_attn=nn.Linear(config.n_embd,3*config.n_embd)
                self.c_proj=nn.Linear(config.n_embd,config.n_embd)
                self.n_embd=config.n_embd
                self.n_head=config.n_head
                self.c_proj.NANOGPT_SCALE_INIT=1
                self.register_buffer("bias",torch.tril(torch.ones(config.block_size,config.block_size)).view(1,1,config.block_size,config.block_size))
        def forward(self,x):
                B,T,C=x.shape
                qkv=self.c_attn(x)
                q,k,v=qkv.split(self.n_embd,dim=2)
                q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
                k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
                v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
                
                # att=(q@k.transpose(-2,-1))*(1/math.sqrt(k.size(-1)))
                # att=att.masked_fill(self.bias[:,:,:T,:T]==0,float("-inf"))
                # att=F.softmax(att,dim=-1)
                # y=att@v
                y=F.scaled_dot_product_attention(q,k,v,is_causal=True)
                
                y=y.transpose(1,2).contiguous().view(B,T,C)
                y=self.c_proj(y)
                return y

class MLP(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.c_fc=nn.Linear(config.n_embd,4*config.n_embd)
                self.gelu=nn.GELU(approximate="tanh")
                self.c_proj=nn.Linear(4*config.n_embd,config.n_embd)
                self.c_proj.NANOGPT_SCALE_INIT=1

        def forward(self,x):
                x=self.c_fc(x)
                x=self.gelu(x)
                x=self.c_proj(x)
                return x

class Block(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.config=config
                self.ln_1=nn.LayerNorm(config.n_embd)
                self.attn=CausalAttention(config)
                self.ln_2=nn.LayerNorm(config.n_embd)
                self.mlp=MLP(config)
        def forward(self,x):
                x=x+self.attn(self.ln_1(x))
                x=x+self.mlp(self.ln_2(x))
                return x

@dataclass
class GPTConfig:
        vocab_size:int=50257
        block_size:int=1024
        n_head:int=12
        n_layer:int=12
        n_embd:int=768
class GPT(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.config=config
                self.transformer=ModuleDict(dict(
                        wte=nn.Embedding(config.vocab_size,config.n_embd),
                        wpe=nn.Embedding(config.block_size,config.n_embd),
                        h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
                        ln_f=nn.LayerNorm(config.n_embd),
                ))
                self.lm_head=nn.Linear(config.n_embd,config.vocab_size,bias=False)
                self.transformer.wte.weight=self.lm_head.weight
                self.apply(self.init_weights)
        def init_weights(self,module):
               std=0.02
               if isinstance(module,nn.Linear):
                      if hasattr(module, 'NANOGPT_SCALE_INIT'):
                                std*=(2*self.config.n_layer)**-0.5
                      torch.nn.init.normal_(module.weight,mean=0.0,std=std)
                      if module.bias is not None:
                          torch.nn.init.zeros_(module.bias)
               if isinstance(module,nn.Embedding):
                        torch.nn.init.normal_(module.weight,mean=0.0,std=std)
        def forward(self, idx, targets=None):
                B, T = idx.size()
                assert T <= self.config.block_size, f"Cannot forward sequence of length {T},block size is only {self.config.block_size}"
                pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
                pos_emb = self.transformer.wpe(pos)
                tok_emb = self.transformer.wte(idx)
                x = tok_emb + pos_emb
                for block in self.transformer.h:
                        x = block(x)
                x = self.transformer.ln_f(x)
                logits = self.lm_head(x)
                loss = None
                if targets is not None:
                        loss = F.cross_entropy(logits.view(-1, logits.size(-1)),targets.view(-1))
                return logits, loss
        def configure_optimizers(self,weight_decay,learning_rate):
                param_dict={pn:p for pn,p in self.named_parameters()}
                param_dict={pn:p for pn,p in param_dict.items() if p.requires_grad}
                decay_params = [p for n,p in param_dict.items() if p.dim() >= 2]
                no_decay_params = [p for n,p in param_dict.items() if p.dim() < 2]
                optim_groups = [
                        {"params": decay_params, "weight_decay": weight_decay},
                        {"params": no_decay_params, "weight_decay": 0.0},
                ]
                num_decay_params = sum(p.numel() for p in decay_params)
                num_no_decay_params = sum(p.numel() for p in no_decay_params)
                print(f"num decay params: {num_decay_params}, num no decay params: {num_no_decay_params}")
                fused_available='fused' in inspect.signature(torch.optim.AdamW).parameters
                use_fused = fused_available and torch.cuda.is_available()
                if use_fused:
                        print(f"using fused AdamW: {use_fused}")
                optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8, fused=use_fused)
                return optimizer
        @classmethod
        def from_pretrained(cls, model_type):
                """Loads pretrained GPT-2 model weights from huggingface"""
                assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
                print("loading weights from pretrained gpt: %s" % model_type)
                config_args = {
                        'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
                        'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
                        'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
                        'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
                }[model_type]
                config_args['vocab_size']=50257
                config_args['block_size']=1024
                config = GPTConfig(**config_args)
                model = GPT(config)
                sd = model.state_dict()
                sd_keys = sd.keys()
                sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')]
                model_hf = GPT2LMHeadModel.from_pretrained(model_type)
                sd_hf = model_hf.state_dict()
                sd_keys_hf = sd_hf.keys()
                sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')]
                sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')]
                transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
                assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
                for k in sd_keys_hf:
                        if any(k.endswith(w) for w in transposed):
                                assert sd_hf[k].shape[::-1]==sd[k].shape
                                with torch.no_grad():
                                        sd[k].copy_(sd_hf[k].t())
                        else:
                                assert sd_hf[k].shape == sd[k].shape
                                with torch.no_grad():
                                        sd[k].copy_(sd_hf[k])
                return model

# def load_tokens(filename):
#     npt = np.load(filename)
#     npt = npt.astype(np.int32)
#     ptt = torch.tensor(npt, dtype=torch.long)
#     return ptt

class DataLoaderLite:
    def __init__(self, B, T, process_rank=0, num_processes=1, split="train"):
        self.B = B
        self.T = T
        self.current_position = 0
        if split=="train":
                filepath="/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/train_tokens_1.bin"
        else:
                filepath="/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/val_tokens_1.bin"
        with open(filepath, "rb") as f:
                tokens_np= np.frombuffer(f.read(),dtype=np.uint16)
        self.tokens = torch.tensor(tokens_np,dtype=torch.long)

        #IF WE WANT TO USE MULTIPLE PROCESSES, WE CAN USE THIS CODE
        # self.process_rank = process_rank
        # self.num_processes = num_processes
        # assert split in {'train', 'val'}
        # data_root = "/mnt/d/AI/GPT-2/fineweb_data/tokenized"
        # shards = os.listdir(data_root)
        # shards = [s for s in shards if split in s]
        # shards = sorted(shards)
        # shards = [os.path.join(data_root, s) for s in shards]
        # self.shards = shards
        # assert len(shards) > 0, f"no shards found for split {split}"
        # if self.process_rank == 0:
        #     print(f"found {len(shards)} shards for split {split}")
        # self.reset()

#     def reset(self):
#         self.current_shard = 0
#         self.tokens = load_tokens(self.shards[self.current_shard])
#         self.current_position = self.B * self.T * self.process_rank

    def next_batch(self):
        B, T = self.B, self.T
        if self.current_position + (B * T + 1) > len(self.tokens):
                self.current_position = 0
        buf = self.tokens[self.current_position : self.current_position+B*T+1]
        x = (buf[:-1]).view(B, T) # inputs
        y = (buf[1:]).view(B, T) # targets
        # advance the position in the tensor
        self.current_position += B * T # * self.num_processes
        # if loading the next batch would be out of bounds, advance to next shard
        # if self.current_position + (B * T * self.num_processes + 1) > len(self.tokens):
        #     self.current_shard = (self.current_shard + 1) % len(self.shards)
        #     self.tokens = load_tokens(self.shards[self.current_shard])
        #     self.current_position = B * T * self.process_rank
        return x, y

B=16
T=1024
batch_size=65536
assert batch_size%(B*T)==0, "batch size must be divisible by B*T"
grad_accum_steps=batch_size//(B*T)
print(f"batch_size={batch_size}")
print(f"using grad_accum_steps={grad_accum_steps}")

max_lr=5e-4
min_lr=max_lr*0.1
warmup_steps=210
max_steps=5721
def lr_schedule(it):
        if it<warmup_steps:
                return max_lr*(it+1)/warmup_steps
        if it>max_steps:
                return min_lr
        decay_ratio=(it-warmup_steps)/(max_steps-warmup_steps)
        coeff=0.5*(1+math.cos(math.pi*decay_ratio))
        return min_lr+(max_lr-min_lr)*coeff

torch.set_float32_matmul_precision('high')
train_loader=DataLoaderLite(B=16, T=1024)
val_loader=DataLoaderLite(B=16, T=1024, split="val")

state_dict=torch.load("/mnt/d/AI/GPT-2/model_step_1905.pt")
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model=GPT(GPTConfig(vocab_size=50304))
model.load_state_dict(state_dict)
model.to("cuda")
model=torch.compile(model)

optimizer=model.configure_optimizers(weight_decay=0.1,learning_rate=5e-4)
best_val_loss=float("inf")

for step in range(max_steps):
        time1=time.time()
        optimizer.zero_grad()
        loss_accum=0.0
        for micro_step in range(grad_accum_steps):
                x,y=train_loader.next_batch()
                x,y=x.to("cuda"),y.to("cuda")
                with torch.autocast(device_type="cuda",dtype=torch.bfloat16):
                        logits,loss=model(x,y)
                loss=loss/grad_accum_steps
                loss_accum += loss.detach()
                loss.backward()
        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)               
        lr=lr_schedule(step)
        for param_group in optimizer.param_groups:
               param_group["lr"]=lr
        optimizer.step()
        if step % 10 == 0:
                val_loss = 0.0
                for _ in range(10):
                        x_val, y_val = val_loader.next_batch()
                        x_val, y_val = x_val.to("cuda"), y_val.to("cuda")
                        with torch.no_grad():
                                _, loss_val = model(x_val, y_val)
                        val_loss += loss_val.item()
                print(f"val_loss: {val_loss/10:.4f}")
                if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        torch.save(model.state_dict(), f"model_best.pt")
                        print(f"New best model saved with val_loss: {best_val_loss/10:.4f}")
                        
        time2=time.time()
        torch.cuda.synchronize()
        tokens_per_sec = (train_loader.B * train_loader.T) / (time2 - time1)
        t=(time2-time1)*1000
        print(f"step {step} | loss {loss_accum.item():.4f} | norm {norm.item():.4f} | lr:{lr} | time: {t:.4f} ms")

/home/awais/miniconda3/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


batch_size=65536
using grad_accum_steps=4
num decay params: 124354560, num no decay params: 121344
using fused AdamW: True


W0426 16:26:35.382000 2635 site-packages/torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


val_loss: 4.3277
New best model saved with val_loss: 4.3277
step 0 | loss 4.3942 | norm 0.5225 | lr:2.380952380952381e-06 | time: 44008.2810 ms
step 1 | loss 4.3349 | norm 0.4555 | lr:4.761904761904762e-06 | time: 4291.3773 ms
step 2 | loss 4.2969 | norm 0.4610 | lr:7.142857142857143e-06 | time: 4107.6572 ms
step 3 | loss 4.3965 | norm 0.4456 | lr:9.523809523809525e-06 | time: 4129.7946 ms
step 4 | loss 4.3277 | norm 0.4383 | lr:1.1904761904761905e-05 | time: 4107.4507 ms
step 5 | loss 4.3516 | norm 0.4376 | lr:1.4285714285714285e-05 | time: 4112.0813 ms
step 6 | loss 4.3748 | norm 0.4528 | lr:1.6666666666666667e-05 | time: 4130.3267 ms
step 7 | loss 4.4163 | norm 0.4836 | lr:1.904761904761905e-05 | time: 4112.7880 ms
step 8 | loss 4.3454 | norm 0.4119 | lr:2.142857142857143e-05 | time: 4107.8269 ms
step 9 | loss 4.4458 | norm 0.4060 | lr:2.380952380952381e-05 | time: 4111.8774 ms
val_loss: 4.4695
step 10 | loss 4.4100 | norm 0.4562 | lr:2.619047619047619e-05 | time: 19747.8514 ms
step

KeyboardInterrupt: 

In [1]:
#Trained on fineweb-edu-350B-0,1,2 .Parquets (2nd half)
import torch
from torch.nn import functional as F
from torch.nn import ModuleDict
from dataclasses import dataclass
from transformers import GPT2LMHeadModel
import torch.nn as nn
import inspect
import math
import time
import numpy as np
import os

class CausalAttention(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.c_attn=nn.Linear(config.n_embd,3*config.n_embd)
                self.c_proj=nn.Linear(config.n_embd,config.n_embd)
                self.n_embd=config.n_embd
                self.n_head=config.n_head
                self.c_proj.NANOGPT_SCALE_INIT=1
                self.register_buffer("bias",torch.tril(torch.ones(config.block_size,config.block_size)).view(1,1,config.block_size,config.block_size))
        def forward(self,x):
                B,T,C=x.shape
                qkv=self.c_attn(x)
                q,k,v=qkv.split(self.n_embd,dim=2)
                q=q.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
                k=k.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
                v=v.view(B,T,self.n_head,C//self.n_head).transpose(1,2)
                
                # att=(q@k.transpose(-2,-1))*(1/math.sqrt(k.size(-1)))
                # att=att.masked_fill(self.bias[:,:,:T,:T]==0,float("-inf"))
                # att=F.softmax(att,dim=-1)
                # y=att@v
                y=F.scaled_dot_product_attention(q,k,v,is_causal=True)
                
                y=y.transpose(1,2).contiguous().view(B,T,C)
                y=self.c_proj(y)
                return y

class MLP(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.c_fc=nn.Linear(config.n_embd,4*config.n_embd)
                self.gelu=nn.GELU(approximate="tanh")
                self.c_proj=nn.Linear(4*config.n_embd,config.n_embd)
                self.c_proj.NANOGPT_SCALE_INIT=1

        def forward(self,x):
                x=self.c_fc(x)
                x=self.gelu(x)
                x=self.c_proj(x)
                return x

class Block(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.config=config
                self.ln_1=nn.LayerNorm(config.n_embd)
                self.attn=CausalAttention(config)
                self.ln_2=nn.LayerNorm(config.n_embd)
                self.mlp=MLP(config)
        def forward(self,x):
                x=x+self.attn(self.ln_1(x))
                x=x+self.mlp(self.ln_2(x))
                return x

@dataclass
class GPTConfig:
        vocab_size:int=50257
        block_size:int=1024
        n_head:int=12
        n_layer:int=12
        n_embd:int=768
class GPT(nn.Module):
        def __init__(self,config):
                super().__init__()
                self.config=config
                self.transformer=ModuleDict(dict(
                        wte=nn.Embedding(config.vocab_size,config.n_embd),
                        wpe=nn.Embedding(config.block_size,config.n_embd),
                        h=nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
                        ln_f=nn.LayerNorm(config.n_embd),
                ))
                self.lm_head=nn.Linear(config.n_embd,config.vocab_size,bias=False)
                self.transformer.wte.weight=self.lm_head.weight
                self.apply(self.init_weights)
        def init_weights(self,module):
               std=0.02
               if isinstance(module,nn.Linear):
                      if hasattr(module, 'NANOGPT_SCALE_INIT'):
                                std*=(2*self.config.n_layer)**-0.5
                      torch.nn.init.normal_(module.weight,mean=0.0,std=std)
                      if module.bias is not None:
                          torch.nn.init.zeros_(module.bias)
               if isinstance(module,nn.Embedding):
                        torch.nn.init.normal_(module.weight,mean=0.0,std=std)
        def forward(self, idx, targets=None):
                B, T = idx.size()
                assert T <= self.config.block_size, f"Cannot forward sequence of length {T},block size is only {self.config.block_size}"
                pos = torch.arange(0, T, dtype=torch.long, device=idx.device)
                pos_emb = self.transformer.wpe(pos)
                tok_emb = self.transformer.wte(idx)
                x = tok_emb + pos_emb
                for block in self.transformer.h:
                        x = block(x)
                x = self.transformer.ln_f(x)
                logits = self.lm_head(x)
                loss = None
                if targets is not None:
                        loss = F.cross_entropy(logits.view(-1, logits.size(-1)),targets.view(-1))
                return logits, loss
        def configure_optimizers(self,weight_decay,learning_rate):
                param_dict={pn:p for pn,p in self.named_parameters()}
                param_dict={pn:p for pn,p in param_dict.items() if p.requires_grad}
                decay_params = [p for n,p in param_dict.items() if p.dim() >= 2]
                no_decay_params = [p for n,p in param_dict.items() if p.dim() < 2]
                optim_groups = [
                        {"params": decay_params, "weight_decay": weight_decay},
                        {"params": no_decay_params, "weight_decay": 0.0},
                ]
                num_decay_params = sum(p.numel() for p in decay_params)
                num_no_decay_params = sum(p.numel() for p in no_decay_params)
                print(f"num decay params: {num_decay_params}, num no decay params: {num_no_decay_params}")
                fused_available='fused' in inspect.signature(torch.optim.AdamW).parameters
                use_fused = fused_available and torch.cuda.is_available()
                if use_fused:
                        print(f"using fused AdamW: {use_fused}")
                optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=(0.9, 0.95), eps=1e-8, fused=use_fused)
                return optimizer
        @classmethod
        def from_pretrained(cls, model_type):
                """Loads pretrained GPT-2 model weights from huggingface"""
                assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
                print("loading weights from pretrained gpt: %s" % model_type)
                config_args = {
                        'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
                        'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
                        'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
                        'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
                }[model_type]
                config_args['vocab_size']=50257
                config_args['block_size']=1024
                config = GPTConfig(**config_args)
                model = GPT(config)
                sd = model.state_dict()
                sd_keys = sd.keys()
                sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')]
                model_hf = GPT2LMHeadModel.from_pretrained(model_type)
                sd_hf = model_hf.state_dict()
                sd_keys_hf = sd_hf.keys()
                sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')]
                sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')]
                transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
                assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
                for k in sd_keys_hf:
                        if any(k.endswith(w) for w in transposed):
                                assert sd_hf[k].shape[::-1]==sd[k].shape
                                with torch.no_grad():
                                        sd[k].copy_(sd_hf[k].t())
                        else:
                                assert sd_hf[k].shape == sd[k].shape
                                with torch.no_grad():
                                        sd[k].copy_(sd_hf[k])
                return model

# def load_tokens(filename):
#     npt = np.load(filename)
#     npt = npt.astype(np.int32)
#     ptt = torch.tensor(npt, dtype=torch.long)
#     return ptt

class DataLoaderLite:
    def __init__(self, B, T, process_rank=0, num_processes=1, split="train"):
        self.B = B
        self.T = T
        self.current_position = 0
        if split=="train":
                filepath="/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/train_tokens_2.bin"
        else:
                filepath="/mnt/d/AI/GPT-2/fineweb_data/tokenized 2/val_tokens_2.bin"
        with open(filepath, "rb") as f:
                tokens_np= np.frombuffer(f.read(),dtype=np.uint16)
        self.tokens = torch.tensor(tokens_np,dtype=torch.long)

        #IF WE WANT TO USE MULTIPLE PROCESSES, WE CAN USE THIS CODE
        # self.process_rank = process_rank
        # self.num_processes = num_processes
        # assert split in {'train', 'val'}
        # data_root = "/mnt/d/AI/GPT-2/fineweb_data/tokenized"
        # shards = os.listdir(data_root)
        # shards = [s for s in shards if split in s]
        # shards = sorted(shards)
        # shards = [os.path.join(data_root, s) for s in shards]
        # self.shards = shards
        # assert len(shards) > 0, f"no shards found for split {split}"
        # if self.process_rank == 0:
        #     print(f"found {len(shards)} shards for split {split}")
        # self.reset()

#     def reset(self):
#         self.current_shard = 0
#         self.tokens = load_tokens(self.shards[self.current_shard])
#         self.current_position = self.B * self.T * self.process_rank

    def next_batch(self):
        B, T = self.B, self.T
        if self.current_position + (B * T + 1) > len(self.tokens):
                self.current_position = 0
        buf = self.tokens[self.current_position : self.current_position+B*T+1]
        x = (buf[:-1]).view(B, T) # inputs
        y = (buf[1:]).view(B, T) # targets
        # advance the position in the tensor
        self.current_position += B * T # * self.num_processes
        # if loading the next batch would be out of bounds, advance to next shard
        # if self.current_position + (B * T * self.num_processes + 1) > len(self.tokens):
        #     self.current_shard = (self.current_shard + 1) % len(self.shards)
        #     self.tokens = load_tokens(self.shards[self.current_shard])
        #     self.current_position = B * T * self.process_rank
        return x, y

B=16
T=1024
batch_size=65536
assert batch_size%(B*T)==0, "batch size must be divisible by B*T"
grad_accum_steps=batch_size//(B*T)
print(f"batch_size={batch_size}")
print(f"using grad_accum_steps={grad_accum_steps}")

max_lr=1e-3
min_lr=max_lr*0.1
warmup_steps=70
max_steps=1907

def lr_schedule(it):
        if it<warmup_steps:
                return max_lr*(it+1)/warmup_steps
        if it>max_steps:
                return min_lr
        decay_ratio=(it-warmup_steps)/(max_steps-warmup_steps)
        coeff=0.5*(1+math.cos(math.pi*decay_ratio))
        return min_lr+(max_lr-min_lr)*coeff

torch.set_float32_matmul_precision('high')
train_loader=DataLoaderLite(B=16, T=1024)
val_loader=DataLoaderLite(B=16, T=1024, split="val")

state_dict=torch.load("/mnt/d/AI/GPT-2/model_best.pt")
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
model=GPT(GPTConfig(vocab_size=50304))
model.load_state_dict(state_dict)
model.to("cuda")
model=torch.compile(model)

optimizer=model.configure_optimizers(weight_decay=0.1,learning_rate=5e-4)
best_val_loss=float("inf")

for step in range(max_steps):
        time1=time.time()
        optimizer.zero_grad()
        loss_accum=0.0
        for micro_step in range(grad_accum_steps):
                x,y=train_loader.next_batch()
                x,y=x.to("cuda"),y.to("cuda")
                with torch.autocast(device_type="cuda",dtype=torch.bfloat16):
                        logits,loss=model(x,y)
                loss=loss/grad_accum_steps
                loss_accum += loss.detach()
                loss.backward()
        norm=torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)               
        lr=lr_schedule(step)
        for param_group in optimizer.param_groups:
               param_group["lr"]=lr
        optimizer.step()
        if step % 20 == 0:
                val_loss = 0.0
                for _ in range(10):
                        x_val, y_val = val_loader.next_batch()
                        x_val, y_val = x_val.to("cuda"), y_val.to("cuda")
                        with torch.no_grad():
                                _, loss_val = model(x_val, y_val)
                        val_loss += loss_val.item()
                print(f"val_loss: {val_loss/10:.4f}")
                if val_loss < best_val_loss:
                        best_val_loss = val_loss
                        torch.save(model.state_dict(), f"model_best_final.pt")
                        print(f"New best model saved with val_loss: {best_val_loss/10:.4f}")
                        
        time2=time.time()
        torch.cuda.synchronize()
        tokens_per_sec = (train_loader.B * train_loader.T) / (time2 - time1)
        t=(time2-time1)*1000
        print(f"step {step} | loss {loss_accum.item():.4f} | norm {norm.item():.4f} | lr:{lr} | time: {t:.4f} ms")

/home/awais/miniconda3/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


batch_size=65536
using grad_accum_steps=4
num decay params: 124354560, num no decay params: 121344
using fused AdamW: True


W0426 20:44:26.071000 1454 site-packages/torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


val_loss: 3.9704
New best model saved with val_loss: 3.9704
step 0 | loss 4.0217 | norm 0.4657 | lr:1.4285714285714285e-05 | time: 45013.9334 ms
step 1 | loss 4.0838 | norm 0.4643 | lr:2.857142857142857e-05 | time: 4283.4132 ms
step 2 | loss 4.0027 | norm 0.4026 | lr:4.2857142857142856e-05 | time: 4293.2966 ms
step 3 | loss 4.0230 | norm 0.3949 | lr:5.714285714285714e-05 | time: 4286.3050 ms
step 4 | loss 4.0793 | norm 0.4269 | lr:7.142857142857143e-05 | time: 4200.0008 ms
step 5 | loss 3.9640 | norm 0.4469 | lr:8.571428571428571e-05 | time: 4125.3283 ms
step 6 | loss 4.0979 | norm 0.4038 | lr:0.0001 | time: 4131.1522 ms
step 7 | loss 4.0378 | norm 0.4788 | lr:0.00011428571428571428 | time: 4131.6586 ms
step 8 | loss 4.0059 | norm 0.4160 | lr:0.00012857142857142858 | time: 4132.6954 ms
step 9 | loss 4.0931 | norm 0.4842 | lr:0.00014285714285714287 | time: 4129.6804 ms
step 10 | loss 3.9338 | norm 0.4234 | lr:0.00015714285714285713 | time: 4158.9563 ms
step 11 | loss 4.1189 | norm 0.464